In [1]:
import os
import tensorflow as tf

IMAGE_SIZE = (128, 128)
BATCH_SIZE = 64
DATA_DIR = './Extracted_Trash_Images'

# Data for training,validation and testing were taken from https://huggingface.co/datasets/TrashBusters/combined

# loading the training set, src=https://www.tensorflow.org/tutorials/load_data/images
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'train'),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)
# loading the validation set
val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'validation'),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
print("\nDetected Trash Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)


📦 Initializing data assembly line...
Found 20759 files belonging to 6 classes.
Found 4454 files belonging to 6 classes.

Detected Trash Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


In [115]:
num_classes = 6

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = True

In [116]:
model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./127.5, offset=-1),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)
history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stopping]
)

Epoch 1/20
325/325 ━━━━━━━━━━━━━━━━━━━━ 129s 379ms/step - accuracy: 0.4209 - loss: 1.5138 - val_accuracy: 0.6388 - val_loss: 1.0242
Epoch 2/20
325/325 ━━━━━━━━━━━━━━━━━━━━ 126s 386ms/step - accuracy: 0.6566 - loss: 0.9673 - val_accuracy: 0.7335 - val_loss: 0.7696
Epoch 3/20
325/325 ━━━━━━━━━━━━━━━━━━━━ 122s 375ms/step - accuracy: 0.7340 - loss: 0.7606 - val_accuracy: 0.7678 - val_loss: 0.6497
Epoch 4/20
325/325 ━━━━━━━━━━━━━━━━━━━━ 123s 379ms/step - accuracy: 0.7719 - loss: 0.6470 - val_accuracy: 0.7959 - val_loss: 0.5809
Epoch 5/20
325/325 ━━━━━━━━━━━━━━━━━━━━ 123s 377ms/step - accuracy: 0.8058 - loss: 0.5587 - val_accuracy: 0.8134 - val_loss: 0.5373
Epoch 6/20
325/325 ━━━━━━━━━━━━━━━━━━━━ 120s 369ms/step - accuracy: 0.8297 - loss: 0.4924 - val_accuracy: 0.8204 - val_loss: 0.5045
Epoch 7/20
325/325 ━━━━━━━━━━━━━━━━━━━━ 156s 481ms/step - accuracy: 0.8475 - loss: 0.4412 - val_accuracy: 0.8318 - val_loss: 0.4776
Epoch 8/20
325/325 ━━━━━━━━━━━━━━━━━━━━ 127s 389ms/step - accuracy: 0.8664 -

In [117]:
#Testing the model on test_data
test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_DIR, 'test'),
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loss, test_accuracy = model.evaluate(test_ds)

print("Test Accuracy:", test_accuracy)
print("Test Loss:", test_loss)

Found 4448 files belonging to 6 classes.
70/70 ━━━━━━━━━━━━━━━━━━━━ 5s 67ms/step - accuracy: 0.8833 - loss: 0.3794
Test Accuracy: 0.8833183646202087
Test Loss: 0.3793914020061493


In [119]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

y_true = []
y_pred = []

for images, labels in test_ds:
    predictions = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(predictions, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(cm)

# Classification report
print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=test_ds.class_names
))

Confusion Matrix:
[[672   2  11  17  10  13]
 [  0 698  17   3  26   7]
 [  7   9 678  24  10  11]
 [ 37   4  16 631  31  25]
 [  5  39  26  46 602  29]
 [ 15   7  16  23  33 648]]

Classification Report:
              precision    recall  f1-score   support

   cardboard       0.91      0.93      0.92       725
       glass       0.92      0.93      0.92       751
       metal       0.89      0.92      0.90       739
       paper       0.85      0.85      0.85       744
     plastic       0.85      0.81      0.83       747
       trash       0.88      0.87      0.88       742

    accuracy                           0.88      4448
   macro avg       0.88      0.88      0.88      4448
weighted avg       0.88      0.88      0.88      4448



2026-08-29 14:31:21.077949: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [121]:
model.save("waste_classifier.keras")